In [8]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
import re
from queue import SimpleQueue

# Load environment variables from your custom env file
load_dotenv(dotenv_path="groq_api_key.env")

# --- API Configuration ---
USE_API = True  # set to False if you want offline dummy mode
API_KEY = os.getenv("GROQ_API_KEY")

if not API_KEY:
    print("[dummy_agents] No Groq API key found in groq_api_key.env")
    USE_API = False
else:
    print("[dummy_agents] Groq API key found!")

# Groq uses OpenAI-compatible API
BASE_URL = "https://api.groq.com/openai/v1"
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

[dummy_agents] Groq API key found!


In [2]:
problem = "x = 6 * 7 + 5 * 6 + 1 * 4, what is x?"

In [2]:
problem = """
A rectangular garden has a length that is 3 meters more than twice its width.
The perimeter of the garden is 54 meters.
Find the area of the garden.

Final Answer: <area>
"""

In [47]:
problem="""A rectangle has a length that is 4 meters longer than its width. If the perimeter of the rectangle is 28 meters, what is its area?

Final Answer: <area>"""

In [48]:
class MessageBus:
    def __init__(self):
        self.queue = SimpleQueue()

    def send(self, message: dict):
        """Send a JSON message through the bus."""
        json_msg = json.dumps(message, indent=2)
        self.queue.put(json_msg)
        print(f"[MessageBus] Sent message:\n{json_msg}\n")

    def receive(self):
        """Receive a message from the bus."""
        if self.queue.empty():
            return None
        json_msg = self.queue.get()
        message = json.loads(json_msg)
        print(f"[MessageBus] Received message:\n{json.dumps(message, indent=2)}\n")
        return message

In [49]:
import os, re, json
from openai import OpenAI  # Groq OpenAI-compatible client

class Finalizer:
    def __init__(self, bus):
        self.name = "Finalizer"
        self.bus = bus
        api_key = os.getenv("GROQ_API_KEY")
        if api_key and OpenAI:
            self.client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")
            print(f"[{self.name}] Groq API connected.")
        else:
            self.client = None
            print(f"[{self.name}] No API key found. Dummy mode.")

    def solve_problem(self, problem: str):
        """Solve a problem by splitting into subtasks and producing a final answer."""
        # ---------------------- Dummy fallback ----------------------
        if not self.client:
            subtasks = [
                {"step": 1, "subproblem": "6 * 7", "answer": 42},
                {"step": 2, "subproblem": "5 * 6", "answer": 30},
                {"step": 3, "subproblem": "1 * 4", "answer": 4},
            ]
            final_answer = sum(s["answer"] for s in subtasks)
            msg = {
                "sender": self.name,
                "recipient": "Verifier",
                "subtasks": subtasks,
                "final_answer": final_answer,
                "problem": problem
            }
            self.bus.send(msg)
            return msg

        # ---------------------- Groq API mode ----------------------
        prompt = f"""
You are a helpful math assistant.
Break the problem into step-by-step subtasks, solve each one,
and provide the final answer in JSON format.
The JSON should look like this:

{{
  "subtasks": [
    {{"step": 1, "subproblem": "...", "answer": "..."}},
    ...
  ],
  "final_answer": ...
}}

Problem:
{problem}
"""

        response = self.client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        raw_output = response.choices[0].message.content.strip()
        print(f"[{self.name}] Raw Groq output:\n{raw_output}\n")

        # ---------------------- Extract JSON ----------------------
        try:
            match = re.search(r"\{.*\}", raw_output, re.DOTALL)
            parsed_json = json.loads(match.group()) if match else {}
        except Exception as e:
            parsed_json = {"error": str(e), "raw_output": raw_output}

        # Ensure subtasks exist
        subtasks = parsed_json.get("subtasks", [])
        if not subtasks:
            # fallback: create one subtask with full problem
            subtasks = [{"step": 1, "subproblem": problem, "answer": parsed_json.get("final_answer")}]

        cleaned_subtasks = []
        for i, sub in enumerate(subtasks, start=1):
            cleaned_subtasks.append({
                "step": sub.get("step", i),
                "subproblem": sub.get("subproblem", f"Subproblem {i} missing"),
                "answer": sub.get("answer", None)
            })

        # Build the final message for Verifier
        final_msg = {
            "sender": self.name,
            "recipient": "Verifier",
            "problem": problem,
            "subtasks": cleaned_subtasks,
            "final_answer": parsed_json.get("final_answer", parsed_json.get("area"))
        }

        # Send to the message bus
        self.bus.send(final_msg)
        return final_msg


In [50]:
finalizer = Finalizer(bus)

result = finalizer.solve_problem(problem)

print("\n=== FINALIZER OUTPUT ===")
print(json.dumps(result, indent=2))

[Finalizer] Groq API connected.
[Finalizer] Raw Groq output:
To solve this problem, we'll break it down into step-by-step subtasks.

**Step 1: Define the relationship between the width and length of the rectangle.**

Let's denote the width of the rectangle as 'w'. Since the length is 4 meters longer than the width, the length can be expressed as 'w + 4'.

**Step 2: Write an equation for the perimeter of the rectangle.**

The perimeter of a rectangle is given by the formula P = 2(length + width). We know that the perimeter is 28 meters, so we can write the equation as:

28 = 2(w + (w + 4))

**Step 3: Simplify the equation and solve for 'w'.**

First, let's simplify the equation by distributing the 2:

28 = 2w + 2w + 8

Combine like terms:

28 = 4w + 8

Subtract 8 from both sides:

20 = 4w

Divide both sides by 4:

5 = w

**Step 4: Find the length of the rectangle.**

Now that we know the width is 5 meters, we can find the length by adding 4:

length = w + 4
= 5 + 4
= 9

**Step 5: Calcul

In [51]:
class Verifier:
    def __init__(self, bus):
        self.name = "Verifier"
        self.client = None
        api_key = os.getenv("GROQ_API_KEY")
        if api_key:
            from openai import OpenAI
            self.client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")
            print(f"[{self.name}] Groq API connected.")
        else:
            print(f"[{self.name}] No API key found. Dummy mode.")

    def verify_solution(self, finalizer_output: dict):
        import json
        subtasks = finalizer_output.get("subtasks", [])

        if not subtasks:
            print(f"[{self.name}] WARNING: No subtasks found in finalizer output!")
            return {
                "sender": self.name,
                "verdicts": [],
                "overall_correct": False
            }

        verdicts = []
        for subtask in subtasks:
            # Dummy evaluation: compare answer to subproblem (if numeric)
            try:
                is_correct = eval(str(subtask["answer"])) == eval(str(subtask["subproblem"]))
            except:
                is_correct = True  # fallback
            verdicts.append({
                "step": subtask["step"],
                "is_correct": is_correct,
                "feedback": "Looks good!" if is_correct else "Check calculation"
            })

        overall_correct = all(v["is_correct"] for v in verdicts)

        return {
            "sender": self.name,
            "verdicts": verdicts,
            "overall_correct": overall_correct
        }


In [52]:
finalizer = Finalizer(bus)
verifier = Verifier(bus)

finalizer_output = finalizer.solve_problem(problem)
verifier_output = verifier.verify_solution(finalizer_output)

print("\n=== FINALIZER OUTPUT ===")
print(json.dumps(finalizer_output, indent=2))

print("\n=== VERIFIER OUTPUT ===")
print(json.dumps(verifier_output, indent=2))



[Finalizer] Groq API connected.
[Verifier] Groq API connected.
[Finalizer] Raw Groq output:
To solve this problem, we'll break it down into step-by-step subtasks.

**Step 1: Define the relationship between the width and length of the rectangle.**

Let's denote the width of the rectangle as 'w'. Since the length is 4 meters longer than the width, the length can be expressed as 'w + 4'.

**Step 2: Write an equation for the perimeter of the rectangle.**

The perimeter of a rectangle is given by the formula P = 2(length + width). We know the perimeter is 28 meters, so we can write the equation as:

28 = 2((w + 4) + w)

**Step 3: Simplify the equation.**

Combine like terms inside the parentheses:

28 = 2(2w + 4)

Distribute the 2:

28 = 4w + 8

**Step 4: Solve for the width 'w'.**

Subtract 8 from both sides:

20 = 4w

Divide both sides by 4:

5 = w

**Step 5: Find the length of the rectangle.**

Now that we know the width is 5 meters, we can find the length by adding 4:

length = w + 4 = 

In [53]:
class Planner:
    def __init__(self, bus: MessageBus):
        self.name = "Planner"
        self.client = None
        api_key = os.getenv("GROQ_API_KEY")
        if api_key:
            from openai import OpenAI
            self.client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")
            print(f"[{self.name}] Groq API connected.")
        else:
            print(f"[{self.name}] No API key found. Dummy mode.")

    def plan_next_step(self, finalizer_output: dict, verifier_output: dict):
        import json, re

        # Dummy mode: simple rules
        if not self.client:
            print(f"[{self.name}] Dummy planning...")
            if verifier_output.get("overall_correct"):
                return {
                    "sender": self.name,
                    "message": "All subtasks correct. Workflow complete.",
                    "action": "complete",
                    "note": "No further refinement needed."
                }
            else:
                incorrect_steps = [
                    v["step"] for v in verifier_output.get("verdicts", [])
                    if not v.get("is_correct", True)
                ]
                return {
                    "sender": self.name,
                    "message": "Some subtasks incorrect. Requesting refinement.",
                    "action": "refine",
                    "note": f"Steps needing attention: {incorrect_steps or 'unknown'}"
                }

        # GPT planning 
        prompt = f"""
You are a planner agent. Decide next action for the Finalizer based on verifier results.
Return JSON with keys: "action" ("refine" or "complete") and "note".
Finalizer output: {json.dumps(finalizer_output, indent=2)}
Verifier output: {json.dumps(verifier_output, indent=2)}
"""
        response = self.client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        raw_output = response.choices[0].message.content.strip()
        match = re.search(r"\{.*\}", raw_output, re.DOTALL)
        if match:
            parsed_json = json.loads(match.group())
        else:
            parsed_json = {"error": "Could not find JSON", "raw_output": raw_output}

        parsed_json["sender"] = self.name
        return parsed_json


In [54]:
bus = MessageBus()
finalizer = Finalizer(bus)
verifier = Verifier(bus)
planner = Planner(bus)

# Step 1: Finalizer
f_out = finalizer.solve_problem(problem)

# Step 2: Verifier
v_out = verifier.verify_solution(f_out)

# Step 3: Planner
p_out = planner.plan_next_step(f_out, v_out)

# Display outputs cleanly
print("\n=== FINALIZER OUTPUT ===")
print(json.dumps(f_out, indent=2))

print("\n=== VERIFIER OUTPUT ===")
print(json.dumps(v_out, indent=2))

print("\n=== PLANNER OUTPUT ===")
print(json.dumps(p_out, indent=2))

[Finalizer] Groq API connected.
[Verifier] Groq API connected.
[Planner] Groq API connected.
[Finalizer] Raw Groq output:
To solve this problem, we'll break it down into step-by-step subtasks.

**Step 1: Define the relationship between the width and length of the rectangle.**

Let's denote the width of the rectangle as 'w'. Since the length is 4 meters longer than the width, the length can be expressed as 'w + 4'.

**Step 2: Write an equation for the perimeter of the rectangle.**

The perimeter of a rectangle is given by the formula P = 2(length + width). We know the perimeter is 28 meters, so we can write the equation as:

28 = 2((w + 4) + w)

**Step 3: Simplify the equation.**

Combine like terms inside the parentheses:

28 = 2(2w + 4)

Distribute the 2:

28 = 4w + 8

**Step 4: Solve for the width 'w'.**

Subtract 8 from both sides:

20 = 4w

Divide both sides by 4:

5 = w

**Step 5: Find the length of the rectangle.**

Now that we know the width is 5 meters, we can find the length b